# LFP v2.1 — Chẩn đoán, KHÔNG train

Notebook này **không train**. Nó preprocess lại dữ liệu để sinh cột provenance
(`cell_idx` / `temp_mean_c` / `cycle_idx`), rồi chấm điểm checkpoint `v2.1-lfp` **đã có sẵn
trong repo** để trả lời một câu hỏi: **quả pin nào trong tập train đang gây lỗi nhiều nhất.**

Lý do cần biết: `scaler_lfp.pkl` đang ghi nhận `temperature min = 0.0 °C` và
`voltage min = 1.889 V/cell`. Cả hai **bất khả thi về vật lý** — không buồng thí nghiệm nào
chạy ở 0 °C và cell LFP không xuống 1.889 V. Nghĩa là có cell nào đó mang dữ liệu cảm biến
hỏng lọt vào lúc train. Xếp hạng sai số theo cell là cách nhanh nhất tìm ra nó.

**Yêu cầu trước khi chạy**
- Code mới **đã push** lên branch `feat/lfp-v21-multitemp`
- 4 file trong `models/weights/*lfp*` **đã commit** — notebook này không tạo ra chúng
- Settings → Internet: **On**
- Attach dataset Severson (`rickandjoe/mit-battery-degradation-dataset`)

**Không cần GPU.** Toàn bộ chạy trên CPU.

## 1 — Clone repo

In [ ]:
import subprocess, os
BRANCH = 'feat/lfp-v21-multitemp'      # <-- doi cho khop nhanh dang lam
REPO   = '/kaggle/working/ai-module'
url = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('Khong co GITHUB_TOKEN secret -> thu public clone:', e)

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, REPO], check=True)
os.chdir(REPO)
print(subprocess.check_output(['git', 'log', '-1', '--format=%h %ci %s']).decode())

## 2 — Chặn bẫy "code trên GitHub chưa mới"

Ô này dừng ngay nếu bản clone về thiếu phần ghi provenance, thay vì để bạn chạy hết
preprocess rồi mới lỗi ở bước cuối.

In [ ]:
import pathlib

checks = {
    'scripts/preprocess_snl.py': ['load_snl_dir', '--severson-dir', 'BATTERYARCHIVE_COLUMNS',
                                  'MAX_DT_SECONDS_DEFAULT', '--cycle-count-norm',
                                  '--snl-cycle-stride', '--artifact-version',
                                  'temp_mean_c'],
    'scripts/preprocess_lfp.py': ['cycles_to_windows', '_longest_discharge_segment',
                                  'MAX_DISCHARGE_SECONDS', 'return_meta'],
    'scripts/eval_soh_by_temp.py': ['temp_mean_c', 'cell_ids'],
    'scripts/train.py':          ['--feature-scaler-version', '--mamba-out', '--iso-out',
                                  '--balance-bands', '--balance-temp-bins'],
    'src/core/config.py':        ['LFP_CYCLE_COUNT_NORM', 'LFP_NOMINAL_CAPACITY_AH',
                                  'LFP_TEMPERATURE_TRAIN_CLUSTERS', 'SPECTRAL_FEAT_DIM'],
}
missing = []
for path, needles in checks.items():
    p = pathlib.Path(path)
    if not p.exists():
        missing.append(path + ': FILE KHONG TON TAI')
        continue
    text = p.read_text(encoding='utf-8')
    for n in needles:
        if n not in text:
            missing.append(f'{path}: thieu {n!r}')
assert not missing, 'Code tren GitHub CHUA MOI:\n  ' + '\n  '.join(missing)
print('OK - code clone ve dung ban moi')

## 3 — Kiểm tra artifact có sẵn

Notebook này chấm điểm checkpoint chứ không tạo ra nó. Thiếu file ở đây gần như luôn có
nghĩa là weights chưa được commit.

In [ ]:
import os, torch

# Checkpoint KHONG duoc train o notebook nay - no phai di theo repo clone ve.
# Neu thieu, gan nhu chac chan la weights chua duoc commit + push.
need = {
    'models/weights/soh_mamba_v2.1-lfp.pth':      'Mamba SOH LFP',
    'models/weights/isolation_forest_v2.1-lfp.pkl':'IsolationForest LFP',
    'models/weights/scaler_lfp.pkl':               'MinMaxScaler LFP',
    'models/weights/feature_scaler_lfp.pkl':       'StandardScaler LFP',
}
missing = [f'{p}  ({label})' for p, label in need.items() if not os.path.exists(p)]
assert not missing, (
    'Thieu artifact trong repo clone ve:
  ' + '
  '.join(missing) +
    '

Notebook nay KHONG train - no cham diem checkpoint co san.
'
    'Commit + push cac file tren len branch roi chay lai tu o Clone.'
)

ck = torch.load('models/weights/soh_mamba_v2.1-lfp.pth', map_location='cpu', weights_only=False)
print('Checkpoint version :', ck.get('version'))
print('  test_mae  =', ck.get('test_mae'), '%')
print('  test_rmse =', ck.get('test_rmse'), '%')
print('  feat_dim  =', ck.get('feat_dim'), '| input_features =', ck.get('input_features'))
print('
OK - du artifact de cham diem, khong can train lai.')

## 4 — Dependencies

In [ ]:
%pip install -q h5py scipy scikit-learn joblib pandas
import h5py, scipy, sklearn
print('h5py', h5py.__version__, '| scipy', scipy.__version__, '| sklearn', sklearn.__version__)

## 5 — Lấy dữ liệu

In [ ]:
import os, glob, subprocess

# --- Severson ---------------------------------------------------------------
mats = glob.glob('/kaggle/input/**/*batch*.mat', recursive=True)
assert mats, ('Khong thay *batch*.mat. + Add Data -> '
              'rickandjoe/mit-battery-degradation-dataset')
SEVERSON_DIR = os.path.dirname(mats[0])
print('SEVERSON_DIR:', SEVERSON_DIR)
for f in sorted(mats):
    print('   ', os.path.basename(f), '(%.2f GB)' % (os.path.getsize(f) / 1e9))

# --- SNL --------------------------------------------------------------------
SNL_SRC = None
pkls = glob.glob('/kaggle/input/**/SNL_18650_LFP*.pkl', recursive=True)
zips = glob.glob('/kaggle/input/**/SNL.zip', recursive=True)
if pkls:
    SNL_SRC = os.path.dirname(pkls[0]);  print('\nSNL tu dataset attach:', SNL_SRC, f'({len(pkls)} pkl)')
elif zips:
    SNL_SRC = zips[0];                   print('\nSNL tu zip attach:', SNL_SRC)
else:
    SNL_SRC = '/kaggle/working/SNL.zip'
    if not os.path.exists(SNL_SRC):
        print('\nTai SNL.zip tu Zenodo (115 MB)...')
        subprocess.run(['curl', '-sSL', '-o', SNL_SRC,
                        'https://zenodo.org/records/19688272/files/SNL.zip?download=1'], check=True)
    size = os.path.getsize(SNL_SRC)
    assert size > 50e6, (f'SNL.zip chi {size} byte -> tai that bai. '
                         f'Bat Settings -> Internet: On, hoac tu upload dataset.')
    print('   OK: %.1f MB' % (size / 1e6))

import zipfile
if zipfile.is_zipfile(SNL_SRC):
    n = len([x for x in zipfile.ZipFile(SNL_SRC).namelist() if 'LFP' in x and x.endswith('.pkl')])
    assert n >= 15, f'Chi thay {n} file LFP trong zip - file hong?'
    print('   %d cell LFP trong archive' % n)

## 6 — Preprocess

Đây là bước tốn thời gian nhất của notebook này, nhưng vẫn ngắn hơn hẳn train.

Tham số phải **giống hệt** lần sinh ra `v2.1-lfp` (`--cycle-count-norm 4600`,
`--cycle-stride 3`, `--snl-cycle-stride 1`, `--soc-mode cycle`). Lệch một tham số là dữ liệu
đem chấm không còn cùng quy ước với lúc train, và mọi con số phía sau đều vô nghĩa.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess_snl.py \
    --snl-dir "{SNL_SRC}" \
    --severson-dir "{SEVERSON_DIR}" \
    --output-dir data/processed_lfp \
    --cycle-stride 3 --snl-cycle-stride 1 --max-dt-seconds 30 \
    --cycle-count-norm 4600 --artifact-version 2.1-lfp \
    --phase discharge --time-unit minutes --soh-clip 100 --soc-mode cycle

## 7 — Phủ sóng (nhiệt độ × dải SOH)

Xem ô nào rỗng: đó là vùng model chưa từng thấy mẫu nào.

In [ ]:
import torch, numpy as np, collections, joblib

TEMP_IDX = 2   # BASE_FEATURES = [voltage, current, temperature, time]
BANDS = [(100, 105), (95, 100), (90, 95), (85, 90), (80, 85), (70, 80), (50, 70), (0, 50)]

# Doi nhiet do da scale -> °C that, de bang doc duoc
sc = joblib.load('models/weights/scaler_lfp.pkl')['scaler']
lo, hi = sc.data_min_[TEMP_IDX], sc.data_max_[TEMP_IDX]
print('scaler temperature range: [%.2f, %.2f] °C' % (lo, hi))

for split in ['train', 'val', 'test']:
    d = torch.load('data/processed_lfp/%s.pt' % split, weights_only=False)
    X, y = d['X'].numpy(), d['y'].numpy()
    t_c = X[:, :, TEMP_IDX].mean(axis=1) * (hi - lo) + lo      # -> °C
    t_g = np.round(t_c / 5.0) * 5.0                            # gom cum 5 °C
    groups = sorted(set(t_g))
    print('\n=== %s: %d window ===' % (split, len(y)))
    hdr = 'SOH band  ' + ''.join('%9.0fC' % g for g in groups) + '     TONG'
    print(hdr); print('-' * len(hdr))
    for b_lo, b_hi in BANDS:
        m = (y >= b_lo) & (y < b_hi)
        if not m.any():
            continue
        row = '%3d-%3d  ' % (b_lo, b_hi)
        for g in groups:
            n = int((m & (t_g == g)).sum())
            row += '%10d' % n if n else '         .'
        print(row + '%9d' % int(m.sum()))
    tot = '   TONG  ' + ''.join('%10d' % int((t_g == g).sum()) for g in groups)
    print(tot + '%9d' % len(y))
print('\n[i] Dau "." = KHONG CO MAU o o do -> vung mu cua model.')

## 8 — Sai số theo nhiệt độ

`bias@healthy` là phép đo trực tiếp của bug gốc: pin **khoẻ** bị đọc **thấp** đi bao nhiêu
điểm ở nhiệt độ đó. Càng âm thì càng dễ sinh ticket giả.

In [ ]:
import glob, os, re, sys, torch, numpy as np, joblib
sys.path.insert(0, '.')
sys.path.insert(0, 'scripts')
from src.core.config import D_MODEL, D_STATE, INPUT_FEATURES, SPECTRAL_FEAT_DIM
from src.models.soh_predictor import MambaSOHPredictor
from train import evaluate          # dung CHINH ham train.py da dung -> khong lech convention

PREV = {'mae': 1.2899, 'rmse': 1.8935}   # v2.0-lfp, lan train tot nhat
def chk(ok, msg): print(('  [OK] ' if ok else '  [!]  ') + msg)

ck = torch.load('models/weights/soh_mamba_v2.1-lfp.pth', map_location='cpu', weights_only=False)
mae, rmse = ck['test_mae'], ck['test_rmse']

print('=== 1. MAE / RMSE TONG THE ===')
chk(mae  < 2.0, 'test MAE  = %.4f %%   (target < 2.0 | v2.0-lfp = %.4f)' % (mae, PREV['mae']))
chk(rmse < 3.0, 'test RMSE = %.4f %%   (target < 3.0 | v2.0-lfp = %.4f)' % (rmse, PREV['rmse']))

print('\n=== 2. PER-BAND (tu train log) ===')
logs = sorted(glob.glob('logs/training/train_*.log'), key=os.path.getmtime)
if logs:
    for ln in open(logs[-1], encoding='utf-8'):
        if re.search(r'SOH\s+\d+-\d+', ln):
            print('  ', ln.split('INFO')[-1].strip())
else:
    print('  (khong tim thay train log)')

print('\n=== 3. BOC TACH THEO NHIET DO ===')
d = torch.load('data/processed_lfp/test.pt', weights_only=False)
X, Xf, y_t = d['X'], d['X_feat'], d['y']
model = MambaSOHPredictor(input_features=INPUT_FEATURES, d_model=D_MODEL,
                          d_state=D_STATE, feat_dim=SPECTRAL_FEAT_DIM)
model.load_state_dict(ck['model_state_dict'])
res = evaluate(model, X, Xf, y_t, torch.device('cpu'))
pred, y = res['pred'].numpy(), y_t.numpy()

# Chan sai lech convention (model xuat SOH/100, evaluate() nhan x100). Neu so tinh
# lai o day khong khop so trong checkpoint thi moi bang ben duoi deu vo nghia.
chk(abs(res['mae'] - mae) < 0.01,
    'MAE tinh lai = %.4f khop checkpoint %.4f' % (res['mae'], mae))

sc = joblib.load('models/weights/scaler_lfp.pkl')['scaler']
lo, hi = sc.data_min_[2], sc.data_max_[2]
t_c = X[:, :, 2].numpy().mean(axis=1) * (hi - lo) + lo
t_g = np.round(t_c / 5.0) * 5.0

print('  %8s %8s %9s %9s | %11s %13s' % ('temp', 'n', 'MAE', 'bias', 'n(SOH>=92)', 'bias@healthy'))
for g in sorted(set(t_g)):
    m = t_g == g
    if m.sum() < 30:
        continue
    err = pred[m] - y[m]
    h = m & (y >= 92)
    hb = '%+13.3f' % (pred[h] - y[h]).mean() if h.sum() >= 20 else '%13s' % 'n/a'
    print('  %7.0fC %8d %9.3f %+9.3f | %11d %s'
          % (g, m.sum(), np.abs(err).mean(), err.mean(), h.sum(), hb))

worst = max((np.abs(pred[t_g == g] - y[t_g == g]).mean()
             for g in set(t_g) if (t_g == g).sum() >= 30), default=0.0)
chk(worst < 3.0, 'MAE cua cum nhiet TE NHAT = %.3f%% (nen < 3.0)' % worst)
print('\n[i] `bias@healthy` la phep do truc tiep cua bug goc: pin KHOE bi doc THAP di bao nhieu')
print('    diem tai nhiet do do. Con am nhieu (vd -8) = bug van con.')

## 9 — Sai số theo **cell** — phần chính

Ô 8 không làm được việc này: danh tính cell không nằm trong `X`, chỉ có trong cột provenance
mới thêm.

Chạy trên **train** trước, không phải test — cell mang dữ liệu hỏng nằm trong train
(test chỉ có đúng 1 cell SNL).

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/eval_soh_by_temp.py     --data-dir data/processed_lfp     --split train     --weights models/weights/soh_mamba_v2.1-lfp.pth

## 10 — Đối chiếu trên split test

In [ ]:
# Split test de doi chieu. Chi 1 cell SNL nen bang se rat thua - do la ket qua
# dung, khong phai loi: no cho thay chinh gioi han cua split hien tai.
!python scripts/eval_soh_by_temp.py     --data-dir data/processed_lfp     --split test     --weights models/weights/soh_mamba_v2.1-lfp.pth

## 11 — Đọc kết quả thế nào

Trong bảng **"Theo cell"** (đã xếp MAE giảm dần):

| Thấy gì | Nghĩa là gì |
|---|---|
| 1–2 cell có MAE vọt cao hẳn so với phần còn lại | **Ứng viên dữ liệu hỏng.** Đây là thứ cần tìm |
| MAE cao đều ở mọi cell cùng một mức nhiệt | Không phải data bẩn — model thiếu dữ liệu ở vùng nhiệt đó |
| Cột `bias` ở ô 8 trôi theo nhiệt độ | Model đang đọc nhiệt độ như thay cho tuổi pin (confound) |

Nếu ra khả năng 1: lấy tên cell đó, loại khỏi train, preprocess + train lại. Vì scaler được
fit lại nên **phải bump `LFP_MODEL_VERSION`** cùng lúc, nếu không server sẽ chặn nạp và mọi
request `chemistry="LFP"` fail.

Dán bảng kết quả vào chat để đọc cùng nhau.